# LinkedIn Industry Performance

In [1]:
from pathlib import Path

import altair as alt
import attaviz
import numpy as np
import pandas as pd

attaviz.enable()
alt.data_transformers.enable("vegafusion")


def find_project_root(marker="pyproject.toml"):
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "LinkedIn"
PROCESSED_PATH = DATA_PATH / "processed"
PROCESSED_PATH.mkdir(exist_ok=True)

LHR_FILE = (
    DATA_PATH / "LinkedIn Hiring Rate" / "LinkedIn_LHR by Industry SA_Aug2026.xlsx"
)
LHR_VALUE = "LHR (SA)"
BASELINE = 1.0

WEST_AFRICA = ["Ghana", "Nigeria"]
COMPARATORS = ["India", "Kenya", "South Africa"]
COUNTRIES = WEST_AFRICA + COMPARATORS

SOURCE_NOTE = (
    "Source: LinkedIn Economic Graph, seasonally adjusted hiring rate "
    "(release Aug 2026).\n"
    "The index is normalised so that the 2016 monthly average equals 1.0."
)

In [2]:
def tidy_lhr(excel_file, sheet_name, names, countries=None):
    """Read one LHR sheet and return a tidy frame."""
    return (
        pd.read_excel(excel_file, sheet_name=sheet_name, header=3)
        .drop(columns=["Unnamed: 0"])
        .set_axis(names, axis="columns")
        .loc[lambda d: d["Country"].ne("Country")]
        .dropna(subset=names)
        .assign(
            Month=lambda d: pd.to_datetime(d["Month"]),
            Country=lambda d: d["Country"].str.strip(),
            **{LHR_VALUE: lambda d: pd.to_numeric(d[LHR_VALUE])},
        )
        .loc[lambda d: d["Country"].isin(countries) if countries else slice(None)]
        .sort_values(names[:-1])
        .reset_index(drop=True)
    )


def baseline_rule(data, value=BASELINE):
    """A single dashed rule at the 2016 baseline, drawn once per panel."""
    return (
        alt.Chart(data)
        .transform_aggregate(_rows="count()")
        .mark_rule(color=attaviz.REFERENCE, strokeDash=[4, 4])
        .encode(y=alt.datum(value))
    )

In [3]:
TIER_LABELS = [
    "Leading Industries",
    "Growing Industries",
    "Transitioning Industries",
    "Emerging Industries",
]

TIER_QUARTILE = {
    "Leading Industries": "1st Quartile",
    "Growing Industries": "2nd Quartile",
    "Transitioning Industries": "3rd Quartile",
    "Emerging Industries": "4th Quartile",
}


def assign_tiers(ranked_industries):
    """Split industries, already ordered best-first, into four balanced tiers."""
    industries = list(ranked_industries)
    n_tiers = min(len(TIER_LABELS), len(industries))
    if n_tiers == 0:
        return {}
    groups = np.array_split(np.array(industries, dtype=object), n_tiers)
    return {
        industry: TIER_LABELS[i] for i, group in enumerate(groups) for industry in group
    }


def industry_metrics(industry_lhr):
    """Summarise each country-industry series and label it with a tier."""
    metrics = (
        industry_lhr.sort_values("Month")
        .groupby(["Country", "Industry"])
        .agg(
            Average_LHR=(LHR_VALUE, "mean"),
            Max_LHR=(LHR_VALUE, "max"),
            Min_LHR=(LHR_VALUE, "min"),
            Volatility=(LHR_VALUE, "std"),
            Latest_LHR=(LHR_VALUE, "last"),
            First_LHR=(LHR_VALUE, "first"),
            Start_Date=("Month", "min"),
            End_Date=("Month", "max"),
        )
        .reset_index()
        .assign(Growth_Trend=lambda d: d["Latest_LHR"] - d["First_LHR"])
        .drop(columns="First_LHR")
        .sort_values(["Country", "Average_LHR"], ascending=[True, False])
    )
    tiers = (
        metrics.groupby("Country")["Industry"]
        .apply(lambda s: pd.Series(assign_tiers(s)))
        .rename("Tier")
        .reset_index()
        .rename(columns={"level_1": "Industry"})
    )
    return (
        metrics.merge(tiers, on=["Country", "Industry"], how="left")
        .assign(
            Start_Date=lambda d: d["Start_Date"].dt.strftime("%Y-%m"),
            End_Date=lambda d: d["End_Date"].dt.strftime("%Y-%m"),
            Tier=lambda d: pd.Categorical(
                d["Tier"], categories=TIER_LABELS, ordered=True
            ),
        )
        .loc[
            :,
            [
                "Country",
                "Tier",
                "Industry",
                "Average_LHR",
                "Max_LHR",
                "Min_LHR",
                "Volatility",
                "Latest_LHR",
                "Growth_Trend",
                "Start_Date",
                "End_Date",
            ],
        ]
        .sort_values(["Country", "Average_LHR"], ascending=[True, False])
        .reset_index(drop=True)
    )


def plot_industry_tiers(industry_lhr, metrics, country, width=300, height=190):
    """Industry time series for one country, one panel per quartile."""
    tiers = metrics.loc[
        metrics["Country"] == country, ["Industry", "Tier", "Average_LHR"]
    ]
    data = (
        industry_lhr.loc[industry_lhr["Country"] == country]
        .merge(tiers, on="Industry", how="inner")
        .assign(
            Tier=lambda d: d["Tier"].astype(str),
            Panel=lambda d: d["Tier"] + " (" + d["Tier"].map(TIER_QUARTILE) + ")",
            Series=lambda d: (
                d["Industry"] + " (Avg: " + d["Average_LHR"].round(2).astype(str) + ")"
            ),
        )
        .sort_values(["Tier", "Average_LHR"], ascending=[True, False])
    )
    panels = [
        f"{tier} ({TIER_QUARTILE[tier]})"
        for tier in TIER_LABELS
        if tier in set(data["Tier"])
    ]
    series_order = (
        data.drop_duplicates("Series")
        .sort_values("Average_LHR", ascending=False)["Series"]
        .tolist()
    )
    data = data.loc[:, ["Month", LHR_VALUE, "Industry", "Panel", "Series"]]
    highlight = alt.selection_point(fields=["Series"], bind="legend")

    line = (
        alt.Chart(data)
        .mark_line()
        .encode(
            x=alt.X("Month:T", title=None),
            y=alt.Y(f"{LHR_VALUE}:Q", title="Index (2016 = 1.0)"),
            color=alt.Color(
                "Series:N",
                title="Industry",
                sort=series_order,
                legend=alt.Legend(columns=2, symbolLimit=0, labelLimit=320),
            ),
            opacity=alt.when(highlight).then(alt.value(1)).otherwise(alt.value(0.15)),
            tooltip=[
                alt.Tooltip("Industry:N"),
                alt.Tooltip("Month:T", format="%b %Y"),
                alt.Tooltip(f"{LHR_VALUE}:Q", format=".2f"),
            ],
        )
        .add_params(highlight)
    )

    chart = (
        (baseline_rule(data) + line)
        .properties(width=width, height=height)
        .facet(
            facet=alt.Facet("Panel:N", title=None, sort=panels),
            columns=2,
            title=f"LinkedIn Hiring Rate by industry: {country}",
        )
        .resolve_scale(y="independent")
    )
    return attaviz.add_caption(chart, SOURCE_NOTE)

In [4]:
industry_lhr = tidy_lhr(
    LHR_FILE,
    sheet_name="2B - LHR SA by Ctry, Ind",
    names=["Month", "Country", "Industry", LHR_VALUE],
    countries=COUNTRIES,
).assign(Industry=lambda d: d["Industry"].str.strip())

industry_lhr.to_csv(PROCESSED_PATH / "lhr_industry.csv", index=False)

industry_lhr.groupby("Country").agg(
    industries=("Industry", "nunique"),
    months=("Month", "nunique"),
    rows=("Industry", "size"),
)

,industries,months,rows
Country,,,
Ghana,1,115,115
India,19,115,2185
Kenya,7,115,694
Nigeria,10,115,1150
South Africa,18,115,2070


In [13]:
metrics = industry_metrics(industry_lhr)
metrics.to_csv(PROCESSED_PATH / "lhr_industry_metrics.csv", index=False)

In [14]:
metrics.groupby(["Country", "Tier"], observed=True)[
    ["Average_LHR", "Max_LHR", "Min_LHR", "Volatility", "Growth_Trend"]
].mean().round(2)

Average_LHR  Max_LHR  Min_LHR  \
Country      Tier                                                      
Ghana        Leading Industries               1.50     2.82     0.84   
India        Leading Industries               2.56     4.41     0.78   
             Growing Industries               1.79     2.88     0.50   
             Transitioning Industries         1.68     2.75     0.54   
             Emerging Industries              1.54     2.47     0.56   
Kenya        Leading Industries               1.30     1.75     0.78   
             Growing Industries               0.99     1.47     0.43   
             Transitioning Industries         0.93     1.63     0.33   
             Emerging Industries              0.91     1.35     0.39   
Nigeria      Leading Industries               2.59     4.83     0.90   
             Growing Industries               2.23     3.93     0.90   
             Transitioning Industries         1.81     4.41     0.73   
             Emerging Industries              1.38     2.54     0.41   
South Africa Leading Industries               1.19     1.75     0.58   
             Growing Industries               1.02     1.63     0.41   
             Transitioning Industries         0.97     1.39     0.49   
             Emerging Industries              0.90     1.32     0.33   

                                       Volatility  Growth_Trend  
Country      Tier                                                
Ghana        Leading Industries              0.38          0.31  
India        Leading Industries              1.03          1.75  
             Growing Industries              0.59          0.82  
             Transitioning Industries        0.50          0.53  
             Emerging Industries             0.42          0.37  
Kenya        Leading Industries              0.23          0.21  
             Growing Industries              0.20         -0.19  
             Transitioning Industries        0.24         -0.22  
             Emerging Industries             0.20         -0.36  
Nigeria      Leading Industries              1.10          2.63  
             Growing Industries              0.72          1.34  
             Transitioning Industries        0.68          1.21  
             Emerging Industries             0.32          0.57  
South Africa Leading Industries              0.24         -0.12  
             Growing Industries              0.21         -0.28  
             Transitioning Industries        0.19         -0.33  
             Emerging Industries             0.17         -0.37

In [7]:
plot_industry_tiers(industry_lhr, metrics, "Ghana")

alt.VConcatChart(...)

In [8]:
plot_industry_tiers(industry_lhr, metrics, "Nigeria")

alt.VConcatChart(...)

In [9]:
plot_industry_tiers(industry_lhr, metrics, "India")

alt.VConcatChart(...)

In [10]:
plot_industry_tiers(industry_lhr, metrics, "Kenya")

alt.VConcatChart(...)

In [11]:
plot_industry_tiers(industry_lhr, metrics, "South Africa")

alt.VConcatChart(...)